In [1]:
import geopandas as gpd
top10 = gpd.read_file("top_10_localizaciones_biometano.gpkg")
porcino = gpd.read_file("clasificacion_porcino.gpkg")

In [3]:
porcino = gpd.read_file("clasificacion_porcino.gpkg")
print(porcino.columns.tolist())
print(porcino.head(3))

['explotacion', 'tipo', 'especie', 'clasif', 'municipio', 'provincia', 'oca', 'huso', 'longitud_utm', 'latitud_utm', 'sistema_productivo', 'capacidad_productiva', 'criterio_sostenibilidad', 'capacidad_cebo', 'capacidad_lechones', 'capacidad_recria', 'capacidad_cerdas', 'capacidad_reposicion', 'capacidad_verracos', 'capacidad_total', 'x', 'y', 'geometry']
      explotacion                       tipo especie                  clasif  \
0  ES220010000014  Producción y reproducción  Cerdos  Producción de lechones   
1  ES220010000030  Producción y reproducción  Cerdos         Cebo o Cebadero   
2  ES220010000031  Producción y reproducción  Cerdos         Cebo o Cebadero   

  municipio provincia        oca  huso  longitud_utm latitud_utm  ...  \
0    Abiego    HUESCA  BARBASTRO  30.0      744539.0     4672000  ...   
1    Abiego    HUESCA  BARBASTRO  30.0      742740.0     4667000  ...   
2    Abiego    HUESCA  BARBASTRO  30.0      742420.0     4666450  ...   

  capacidad_cebo capacidad_le

In [4]:
import geopandas as gpd
import pandas as pd
import numpy as np

# ── Cargar datos ──────────────────────────────────────────────────
top10 = gpd.read_file("top_10_localizaciones_biometano.gpkg").to_crs("EPSG:25830")
porcino = gpd.read_file("clasificacion_porcino.gpkg").to_crs("EPSG:25830")

# ── Granjas en radio de 10km por celda ───────────────────────────
resultados = []

for _, celda in top10.iterrows():
    centro = celda.geometry.centroid
    buffer = centro.buffer(10_000)
    
    granjas_cerca = porcino[porcino.geometry.within(buffer)].copy()
    
    n_granjas        = len(granjas_cerca)
    cabezas_total    = granjas_cerca["capacidad_total"].sum()
    cabezas_cebo     = granjas_cerca["capacidad_cebo"].sum()
    cabezas_cerdas   = granjas_cerca["capacidad_cerdas"].sum()
    cabezas_lechones = granjas_cerca["capacidad_lechones"].sum()

    resultados.append({
        "cell_id":         celda["cell_id"],
        "score":           celda["score"],
        "n_granjas":       n_granjas,
        "cabezas_total":   cabezas_total,
        "cabezas_cebo":    cabezas_cebo,
        "cabezas_cerdas":  cabezas_cerdas,
        "cabezas_lechones":cabezas_lechones,
    })

df = pd.DataFrame(resultados).sort_values("score", ascending=False)
print("=== GRANJAS Y CABEZAS EN 10KM POR CELDA ===")
print(df.to_string(index=False))

=== GRANJAS Y CABEZAS EN 10KM POR CELDA ===
 cell_id  score  n_granjas  cabezas_total  cabezas_cebo  cabezas_cerdas  cabezas_lechones
   13279  85.77        239       400971.0      349565.0          9820.0           27887.0
   13277  85.66        231       393636.0      342230.0          9820.0           27887.0
   13290  85.60        238       404947.0      353541.0          9820.0           27887.0
   13293  85.46        240       412969.0      361563.0          9820.0           27887.0
   13291  85.20        242       414946.0      360711.0         11809.0           28727.0
   13412  85.01        247       432144.0      376372.0         13064.0           28727.0
   13294  84.94        245       423166.0      368201.0         12539.0           28727.0
   13278  84.90        236       401642.0      350236.0          9820.0           27887.0
   13414  84.89        247       439764.0      383992.0         13064.0           28727.0
   13416  84.75        251       444037.0      388265.0 

In [5]:
# ── Factores de producción por tipo ──────────────────────────────
# Litros de purín/día por cabeza según tipo
PURIN_CEBO    = 4.5   # l/día
PURIN_CERDA   = 16.0  # l/día
PURIN_LECHON  = 1.5   # l/día

# Potencial de biogás por m³ de purín porcino
BIOGÁS_M3_POR_M3_PURIN = 25    # m³ biogás / m³ purín
CH4_CONTENIDO           = 0.60  # 60% metano en biogás
UPGRADE_EFICIENCIA      = 0.95  # eficiencia del upgrading a biometano

# Digestato: ~95% del purín entrada queda como digestato
DIGESTATO_RATIO = 0.95

resultados_viab = []

for _, row in df.iterrows():
    # Purín diario (m³/día) — convertimos litros a m³
    purin_dia = (
        row["cabezas_cebo"]     * PURIN_CEBO    +
        row["cabezas_cerdas"]   * PURIN_CERDA   +
        row["cabezas_lechones"] * PURIN_LECHON
    ) / 1000

    purin_año = purin_dia * 365

    # Biogás y biometano
    biogas_dia     = purin_dia * BIOGÁS_M3_POR_M3_PURIN
    biometano_dia  = biogas_dia * CH4_CONTENIDO * UPGRADE_EFICIENCIA
    biometano_hora = biometano_dia / 24

    # Digestato
    digestato_dia = purin_dia * DIGESTATO_RATIO
    digestato_año = digestato_dia * 365

    resultados_viab.append({
        "cell_id":          row["cell_id"],
        "score":            row["score"],
        "cabezas_total":    row["cabezas_total"],
        "purin_dia_m3":     round(purin_dia, 1),
        "purin_año_m3":     round(purin_año, 0),
        "biometano_Nm3h":   round(biometano_hora, 1),
        "digestato_dia_m3": round(digestato_dia, 1),
        "digestato_año_m3": round(digestato_año, 0),
    })

df_viab = pd.DataFrame(resultados_viab).sort_values("score", ascending=False)
print("=== DIMENSIONAMIENTO DE PLANTA POR CELDA ===")
print(df_viab.to_string(index=False))

=== DIMENSIONAMIENTO DE PLANTA POR CELDA ===
 cell_id  score  cabezas_total  purin_dia_m3  purin_año_m3  biometano_Nm3h  digestato_dia_m3  digestato_año_m3
 13279.0  85.77       400971.0        1772.0      646777.0          1052.1            1683.4          614439.0
 13277.0  85.66       393636.0        1739.0      634730.0          1032.5            1652.0          602993.0
 13290.0  85.60       404947.0        1789.9      653308.0          1062.7            1700.4          620643.0
 13293.0  85.46       412969.0        1826.0      666484.0          1084.2            1734.7          633160.0
 13291.0  85.20       414946.0        1855.2      677160.0          1101.5            1762.5          643302.0
 13412.0  85.01       432144.0        1945.8      710213.0          1155.3            1848.5          674702.0
 13294.0  84.94       423166.0        1900.6      693726.0          1128.5            1805.6          659040.0
 13278.0  84.90       401642.0        1775.0      647880.0         

In [6]:
gasoductos = gpd.read_file("gasoductos_huesca.gpkg").to_crs("EPSG:25830")
from shapely.ops import unary_union

gasoductos_union = unary_union(gasoductos.geometry)

df_viab["dist_gasoducto_m"] = top10.set_index("cell_id").loc[
    df_viab["cell_id"].values, "dist_gasoducto"
].values

COSTE_KM = 2_100_000  # €/km según dato de Calanda

df_viab["dist_gasoducto_km"] = (df_viab["dist_gasoducto_m"] / 1000).round(2)
df_viab["coste_conexion_€"]  = (df_viab["dist_gasoducto_km"] * COSTE_KM).round(0)

print("=== COSTE DE CONEXIÓN AL GASODUCTO ===")
print(df_viab[["cell_id", "score", "dist_gasoducto_km", "coste_conexion_€"]].to_string(index=False))

=== COSTE DE CONEXIÓN AL GASODUCTO ===
 cell_id  score  dist_gasoducto_km  coste_conexion_€
 13279.0  85.77               1.06         2226000.0
 13277.0  85.66               1.18         2478000.0
 13290.0  85.60               1.47         3087000.0
 13293.0  85.46               1.75         3675000.0
 13291.0  85.20               2.16         4536000.0
 13412.0  85.01               3.10         6510000.0
 13294.0  84.94               2.45         5145000.0
 13278.0  84.90               1.88         3948000.0
 13414.0  84.89               4.14         8694000.0
 13416.0  84.75               3.64         7644000.0


In [7]:
CAPEX_POR_NM3H = 5_000  # €/Nm³h — referencia europea plantas purín porcino

df_viab["capex_planta_€"]  = (df_viab["biometano_Nm3h"] * CAPEX_POR_NM3H).round(0)
df_viab["capex_total_€"]   = (df_viab["capex_planta_€"] + df_viab["coste_conexion_€"]).round(0)
df_viab["capex_total_M€"]  = (df_viab["capex_total_€"] / 1_000_000).round(2)

print("=== CAPEX TOTAL POR CELDA ===")
print(df_viab[["cell_id", "score", "biometano_Nm3h", "capex_planta_€", 
               "coste_conexion_€", "capex_total_M€"]].to_string(index=False))

=== CAPEX TOTAL POR CELDA ===
 cell_id  score  biometano_Nm3h  capex_planta_€  coste_conexion_€  capex_total_M€
 13279.0  85.77          1052.1       5260500.0         2226000.0            7.49
 13277.0  85.66          1032.5       5162500.0         2478000.0            7.64
 13290.0  85.60          1062.7       5313500.0         3087000.0            8.40
 13293.0  85.46          1084.2       5421000.0         3675000.0            9.10
 13291.0  85.20          1101.5       5507500.0         4536000.0           10.04
 13412.0  85.01          1155.3       5776500.0         6510000.0           12.29
 13294.0  84.94          1128.5       5642500.0         5145000.0           10.79
 13278.0  84.90          1053.9       5269500.0         3948000.0            9.22
 13414.0  84.89          1175.7       5878500.0         8694000.0           14.57
 13416.0  84.75          1187.1       5935500.0         7644000.0           13.58
